# 08 — Hyperparameter Tuning (Logistic Regression)

In this notebook, we tune the Logistic Regression model selected in `05_model_comparision.ipynb`.

**Goals:**
- Use `GridSearchCV` with Stratified K-Fold cross-validation
- Tune key parameters: `C`, `penalty`, `solver`, `class_weight`
- Evaluate the best model using the threshold selected in `06_threshold_analysis.ipynb` (0.40)
- Save the final tuned model to `artifacts/`

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')

## 2. Load Data & Preprocessor

In [ ]:
# Load the cleaned dataset (same as used in all previous notebooks)
df = pd.read_csv("../../data/processed/telco_clean.csv")

X = df.drop("Churn", axis=1)
y = df["Churn"]

# Load the preprocessor pipeline saved in notebook 02
preprocessor = joblib.load("../../artifacts/preprocessor.pkl")

# Same split as all previous notebooks — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Churn rate in train: {y_train.mean():.2%}")
print(f"Churn rate in test:  {y_test.mean():.2%}")

## 3. Apply Preprocessor

In [ ]:
# Transform using the saved preprocessor (already fit on training data in notebook 02)
X_train_processed = preprocessor.transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"Processed feature shape: {X_train_processed.shape}")

## 4. Define Hyperparameter Grid

Key parameters being tuned:

| Parameter | What it controls |
|---|---|
| `C` | Inverse of regularization strength — smaller = more regularization |
| `penalty` | Type of regularization (l1 shrinks unimportant features to 0, l2 is default) |
| `solver` | Optimization algorithm — must match the penalty type |
| `class_weight` | Whether to penalize misclassifying the minority class (churners) more |

In [ ]:
param_grid = [
    # l2 penalty — works with lbfgs, liblinear, saga
    {
        'penalty': ['l2'],
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['lbfgs', 'liblinear'],
        'class_weight': [None, 'balanced']
    },
    # l1 penalty — only works with liblinear or saga
    {
        'penalty': ['l1'],
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['liblinear', 'saga'],
        'class_weight': [None, 'balanced']
    }
]

print("Parameter grid defined.")

## 5. Run GridSearchCV

We use **Stratified K-Fold** (5 folds) to ensure each fold preserves the churn ratio.

We score on **`recall`** because for churn, **missing an actual churner is more costly** than a false alarm — aligning with the business reasoning in notebook 06.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='recall',       # prioritise catching actual churners
    n_jobs=-1,              # use all CPU cores
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train_processed, y_train)

print("\n✅ Grid search complete.")
print(f"Best CV Recall Score : {grid_search.best_score_:.4f}")
print(f"Best Parameters      : {grid_search.best_params_}")

## 6. Inspect Top Results

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)

top_results = (
    cv_results[
        ['param_C', 'param_penalty', 'param_solver',
         'param_class_weight', 'mean_test_score', 'std_test_score']
    ]
    .sort_values('mean_test_score', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top_results.columns = ['C', 'Penalty', 'Solver', 'Class Weight', 'Mean Recall (CV)', 'Std']
top_results

## 7. Evaluate Best Model on Test Set

We apply the **same threshold of 0.40** decided in `06_threshold_analysis.ipynb`.

In [ ]:
THRESHOLD = 0.40   # from notebook 06

best_model = grid_search.best_estimator_

y_prob_tuned  = best_model.predict_proba(X_test_processed)[:, 1]
y_pred_tuned  = (y_prob_tuned >= THRESHOLD).astype(int)

print(f"Threshold used : {THRESHOLD}")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred_tuned, target_names=['No Churn', 'Churn']))

## 8. Compare Baseline vs Tuned Model

In [ ]:
# Load baseline model from notebook 02
baseline_model = joblib.load("../../artifacts/logreg_model.pkl")

y_prob_base = baseline_model.predict_proba(X_test_processed)[:, 1]
y_pred_base = (y_prob_base >= THRESHOLD).astype(int)

def get_metrics(y_true, y_pred, y_prob):
    return {
        "Precision" : round(precision_score(y_true, y_pred), 4),
        "Recall"    : round(recall_score(y_true, y_pred), 4),
        "F1 Score"  : round(f1_score(y_true, y_pred), 4),
        "ROC-AUC"   : round(roc_auc_score(y_true, y_prob), 4),
    }

comparison = pd.DataFrame({
    "Baseline LR"  : get_metrics(y_test, y_pred_base, y_prob_base),
    "Tuned LR"     : get_metrics(y_test, y_pred_tuned, y_prob_tuned),
}).T

comparison

## 9. Confusion Matrix — Tuned Model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_pred_base, y_pred_tuned],
    ["Baseline LR (threshold=0.40)", "Tuned LR (threshold=0.40)"]
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.tight_layout()
plt.savefig("../../artifacts/confusion_matrix_tuned.png", dpi=150, bbox_inches='tight')
plt.show()

## 10. Regularization Strength vs Recall (Visualisation)

In [ ]:
c_vals    = [0.01, 0.1, 1, 10, 100]
penalties = ['l1', 'l2']

fig, ax = plt.subplots(figsize=(8, 4))

for penalty in penalties:
    recalls = []
    for c in c_vals:
        subset = cv_results[
            (cv_results['param_C'] == c) &
            (cv_results['param_penalty'] == penalty) &
            (cv_results['param_class_weight'] == grid_search.best_params_['class_weight'])
        ]
        if not subset.empty:
            recalls.append(subset['mean_test_score'].max())
        else:
            recalls.append(np.nan)
    ax.plot(c_vals, recalls, marker='o', label=f'penalty={penalty}')

ax.set_xscale('log')
ax.set_xlabel('C (Regularization Strength)')
ax.set_ylabel('Mean CV Recall')
ax.set_title('Regularization Strength vs Recall')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../../artifacts/c_vs_recall.png", dpi=150, bbox_inches='tight')
plt.show()

## 11. Save Tuned Model

In [ ]:
joblib.dump(best_model, "../../artifacts/logreg_tuned_model.pkl")
print("✅ Tuned model saved to artifacts/logreg_tuned_model.pkl")
print(f"   Best params: {grid_search.best_params_}")

## 12. Summary

### What We Did
- Ran `GridSearchCV` over `C`, `penalty`, `solver`, and `class_weight` using **Stratified 5-Fold CV**
- Optimised for **Recall** — consistent with the business priority of catching churners identified in notebook 06
- Applied the **same 0.40 threshold** to both baseline and tuned models for a fair comparison

### Key Takeaways
- Tuning Logistic Regression with the right regularization and class weighting can meaningfully improve recall on imbalanced churn data
- The `class_weight='balanced'` option helps the model pay more attention to the minority churn class without needing oversampling
- The tuned model is saved as `logreg_tuned_model.pkl` and will be used in the Streamlit deployment

### Next Step
→ Build the **Streamlit app** using `logreg_tuned_model.pkl` + `preprocessor.pkl`